In [ ]:
#print the  current directory
import os
print(os.getcwd())

In [ ]:
# === Preprocess NDVI → write NDVI_ALL.csv (and optional per-crop CSVs) ===
import os
import pandas as pd

# ---------- 1) Read long table (chunk-safe) ----------
src = "Data/AT_ET_farm_data.csv"
chunks = pd.read_csv(src, chunksize=10000)
df = pd.concat(chunks, ignore_index=True)

# keep only needed cols
cols = ["CSBID","CNTY","Shape_Area","CDL","NDVI","year","month","CSBACRES"]
df = df[cols].copy()

# robust typing (coerce then cast)
df["CSBID"]      = pd.to_numeric(df["CSBID"], errors="coerce")
df["CDL"]        = pd.to_numeric(df["CDL"], errors="coerce")
df["NDVI"]       = pd.to_numeric(df["NDVI"], errors="coerce")
df["year"]       = pd.to_numeric(df["year"], errors="coerce")
df["month"]      = pd.to_numeric(df["month"], errors="coerce")
df["Shape_Area"] = pd.to_numeric(df["Shape_Area"], errors="coerce")
df["CSBACRES"]   = pd.to_numeric(df["CSBACRES"], errors="coerce")
df["CNTY"]       = df["CNTY"].astype(str).str.strip().str.upper()

# drop rows missing the essentials
df = df.dropna(subset=["CSBID","year","month","NDVI"]).copy()

# final dtypes
df = df.astype({
    "CSBID": "int64",
    "CDL": "Int64",           # allow NA CDL
    "NDVI": "float64",
    "year": "int64",
    "month": "int64",
    "Shape_Area": "float64",
    "CSBACRES": "float64"
})
df.rename(columns={
    "CNTY": "County",
    "Shape_Area": "Shape_area",
    "year": "Year",
    "month": "Month",
    "CSBACRES": "Shape_acers"
}, inplace=True)

# restrict to target years (keeps ALL crops)
YEARS = list(range(2019, 2025))
df = df[df["Year"].isin(YEARS)].copy()

# ---------- 2) Write the unfiltered file (this is what your modeling needs) ----------
os.makedirs("Data", exist_ok=True)
df.to_csv("Data/NDVI_ALL.csv", index=False)
print("Wrote → Data/NDVI_ALL.csv  (unfiltered; use this for season windows)")


'''
# ---------- 3) (Optional) Also write per-crop CSVs for convenience ----------
# Note: if you will model with prev-year months, DO NOT feed these per-crop files
# into the season builder. Use NDVI_ALL.csv instead.

cdl_cotton  = [2, 232, 238, 239]
cdl_corn    = [1, 225, 226, 228, 237, 241]
cdl_wheat   = [22, 23, 24, 26, 225, 230, 234, 236, 238]
cdl_barley  = [21, 233, 235, 237, 254]
cdl_alfalfa = [36]

# Optional: make classes mutually exclusive via a simple priority to avoid
# the same row landing in multiple crop CSVs in the *same year*.
PRIORITY = [
    ("cotton",  set(cdl_cotton)),
    ("corn",    set(cdl_corn)),
    ("wheat",   set(cdl_wheat)),
    ("barley",  set(cdl_barley)),
    ("alfalfa", set(cdl_alfalfa)),
]

pool = df.copy()
dfs  = {}

for name, codes in PRIORITY:
    sel = pool["CDL"].isin(list(codes))
    dfs[name] = pool[sel].copy()
    pool = pool[~sel]  # remove assigned rows → exclusivity

# Write per-crop files (optional; keep for other tasks)
dfs["cotton"].to_csv("Data/NDVI_COTTON.csv", index=False)
dfs["corn"].to_csv("Data/NDVI_CORN.csv", index=False)
dfs["wheat"].to_csv("Data/NDVI_WHEAT.csv", index=False)
dfs["barley"].to_csv("Data/NDVI_BARLEY.csv", index=False)
dfs["alfalfa"].to_csv("Data/NDVI_ALFALFA.csv", index=False)
print("Wrote per-crop CSVs in Data/ (exclusive by priority).")
'''
